In [1]:
import numpy as np
import cv2
from keras.models import load_model
from keras.losses import MeanSquaredError
import queue
import cv2
import threading
import queue
import socket
import cv2

In [ ]:
test_model =load_model(r'model.h5', custom_objects={'mse':MeanSquaredError()}) #prediction model
# ESP32-CAM's IP and port for receiving video frames
ESP32_IP = "111.111.111.11"
ESP32_PORT = 12346

# Laptop's IP and port for sending commands
LAPTOP_IP = "0.0.0.0"  # Listen on all interfaces
LAPTOP_PORT = 12345
result_queue = queue.Queue()

In [3]:
def img_preprocess(img):
    img = img[250:450,:,:]
    img = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
    img = cv2.GaussianBlur(img,  (3, 3), 0)
    img = cv2.resize(img, (200, 66))
    img = img/255
    return img

In [4]:
def receive_frames():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
        server_socket.bind((LAPTOP_IP, ESP32_PORT))
        server_socket.listen()
        print(f"Listening for ESP32-CAM on {LAPTOP_IP}:{ESP32_PORT}")
        while True:
            conn, addr = server_socket.accept()
            with conn:
                print(f"Connected by {addr}")
                while True:
                    # Receive angle (2 bytes)
                    angle_data = conn.recv(2)
                    if not angle_data: break
                    angle = int.from_bytes(angle_data, byteorder='little')
                
                    # Receive frame length (4 bytes) - ESP32 sends it automatically
                    size_data = conn.recv(4)
                    size = int.from_bytes(size_data, byteorder='little')
                
                    # Receive frame data
                    frame_data = b''
                    while len(frame_data) < size:
                        remaining = size - len(frame_data)
                        frame_data += conn.recv(4096 if remaining > 4096 else remaining)
                
                    # Decode JPEG and overlay angle
                    frame = cv2.imdecode(np.frombuffer(frame_data, dtype=np.uint8), cv2.IMREAD_COLOR)
                    if frame is not None:
                      
                        processed_frame=img_preprocess(frame)
                        frame1 = np.array([processed_frame])
                        results1 =float( test_model.predict(frame1))
                        result_queue.put(results1)
                        cv2.putText(frame, f"Predicted Angle: {angle}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                        if cv2.waitKey(1) & 0xFF == ord('q'):
                            break
                        print(results1)
                    cv2.imshow("ESP32-CAM", frame)
                    
    cv2.destroyAllWindows()


                        
                      



In [5]:
# Function to send commands to ESP32-CAM
def send_commands():
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.connect((ESP32_IP, LAPTOP_PORT))
            print(f"Connected to {ESP32_IP}:{LAPTOP_PORT}")
            s.sendall('W'.encode()+b'\n')
            print("Sent 'W' to ESP32-CAM")
           
        
            while True:
                command= result_queue.get()
                command = f"{command:.1f}"
                print(command)
                try:
                    s.sendall(command.encode()+b'\n')
                    print(f"Sent {command} to ESP32-CAM")
                    
                except Exception as e:
                    print(f"Error: {e}")
                    break

In [ ]:
thread_receive_frames = threading.Thread(target=receive_frames) # control the thread in the background 
thread_send_commands = threading.Thread(target=send_commands)

thread_receive_frames.start()
thread_send_commands.start()

thread_receive_frames.join()
thread_send_commands.join()

Listening for ESP32-CAM on 0.0.0.0:12346
Connected to 192.168.1.106:12345
Sent 'W' to ESP32-CAM
Connected by ('192.168.1.106', 58847)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
51.1
Sent 51.1 to ESP32-CAM
51.13705062866211


C:\Users\pc\AppData\Local\Temp\ipykernel_1500\2259081146.py:32: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  results1 =float( test_model.predict(frame1))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
49.4
Error: [WinError 10054] An existing connection was forcibly closed by the remote host
49.38837432861328
Connected to 192.168.1.106:12345
Sent 'W' to ESP32-CAM
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
49.8
Sent 49.8 to ESP32-CAM
49.83135223388672
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
47.2
Sent 47.2 to ESP32-CAM
47.16777420043945
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
45.3
Sent 45.3 to ESP32-CAM
45.3463020324707
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
45.1
Sent 45.1 to ESP32-CAM
45.10157775878906
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
47.3
Sent 47.3 to ESP32-CAM
47.27817916870117
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
48.0
Sent 48.0 to ESP32-CAM
47.962303161621094
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
45.6
Sent 45.6 to ESP32-CAM
45.609195709228516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
45.8
Sent 45.8 to ESP32-CAM
45.84564208984375
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
45.0
Sent 45.0 to ESP32-CAM
45.03021240234375
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
44.

In [ ]:
# with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
#         server_socket.bind((LAPTOP_IP, ESP32_PORT))
#         server_socket.listen()
#         print(f"Listening for ESP32-CAM on {LAPTOP_IP}:{ESP32_PORT}")
#         while True:
#             print(1)
#             conn, addr = server_socket.accept()
#             print(2)
#             with conn:
#                 print(f"Connected by {addr}")
#                 while True:
#                     # print(3)
#                     # Receive angle (2 bytes)
#                     angle_data = conn.recv(2)
#                     if not angle_data:
#                         break
#                     angle = int.from_bytes(angle_data, byteorder='little')
                
#                     # Receive frame length (4 bytes) - ESP32 sends it automatically
#                     size_data = conn.recv(4)
#                     size = int.from_bytes(size_data, byteorder='little')
                
#                     # Receive frame data
#                     frame_data = b''
#                     while len(frame_data) < size:
#                         remaining = size - len(frame_data)
#                         frame_data += conn.recv(4096 if remaining > 4096 else remaining)
                
#                     # Decode JPEG and overlay angle
#                     frame = cv2.imdecode(np.frombuffer(frame_data, dtype=np.uint8), cv2.IMREAD_COLOR)
#                     if frame is not None:
#                         frame=img_preprocess(frame)
#                         frame1 = np.array([frame])
#                         results1 =float( test_model.predict(frame1))
#                         print(results1)
#                         cv2.imshow("ESP32-CAM", frame)